# ECDC EpiPulse Data Accessor - Example Notebook

This notebook demonstrates how to use the EpiPulse accessor to fetch infectious disease surveillance data from the European Centre for Disease Prevention and Control (ECDC).

## Overview

**EpiPulse** is the European surveillance portal for infectious diseases, integrating data from:
- 27 EU Member States
- 3 EEA countries
- WHO European Region (53 countries total)

**Data Coverage:** 50+ infectious diseases

**Website:** https://epipulse.ecdc.europa.eu/

In [ ]:
# Import necessary libraries
import sys
sys.path.insert(0, '../scripts/accessors')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from epipulse import EpiPulseAccessor, get_epipulse_diseases, get_epipulse_countries

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Libraries imported successfully!")

## 1. Initialize EpiPulse Accessor

Note: EpiPulse requires an API key for full access. You can:
1. Set the `EPIPULSE_API_KEY` environment variable
2. Pass the API key directly to the constructor
3. Store it in `~/.nanobot/config/epipulse.json`

For this example, we'll use the accessor without an API key (limited functionality).

In [ ]:
# Initialize the accessor
epipulse = EpiPulseAccessor()

print("✅ EpiPulse accessor initialized")
print(f"Cache directory: {epipulse.cache_dir}")

## 2. Explore Available Data

### 2.1 List Available Diseases

In [ ]:
# Get list of available diseases
diseases = epipulse.get_available_diseases()

print(f"Total diseases available: {len(diseases)}")
print("\nDiseases by category:")
print(diseases['category'].value_counts())

# Show first 15 diseases
diseases.head(15)

In [ ]:
# Filter diseases by category
respiratory_diseases = diseases[diseases['category'] == 'respiratory']
vaccine_preventable = diseases[diseases['category'] == 'vaccine_preventable']

print("Respiratory diseases:")
print(respiratory_diseases[['disease', 'icd10']])

print("\n\nVaccine-preventable diseases:")
print(vaccine_preventable[['disease', 'icd10']])

### 2.2 List Available Countries

In [ ]:
# Get EU countries
eu_countries = epipulse.get_available_countries(region="EU")

print(f"Total EU countries: {len(eu_countries)}")
print("\nFirst 10 EU countries:")
eu_countries.head(10)

In [ ]:
# Get all countries (EU + EEA)
all_countries = epipulse.get_available_countries(region="all")

print(f"Total countries covered: {len(all_countries)}")
print("\nCountries by region:")
print(all_countries['region'].value_counts())

all_countries.tail(10)

## 3. Fetch Surveillance Data

### 3.1 COVID-19 Data Example

In [ ]:
# Fetch COVID-19 data for Germany in 2024
covid_germany = epipulse.get_cases(
    disease="COVID-19",
    country="DE",
    year=2024
)

print(f"Retrieved {len(covid_germany)} records")
print("\nFirst 5 records:")
covid_germany.head()

In [ ]:
# Plot COVID-19 cases over time
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

# Cases
ax1.plot(covid_germany['date'], covid_germany['cases'], marker='o', linewidth=2)
ax1.set_title('COVID-19 Cases - Germany 2024', fontsize=14, fontweight='bold')
ax1.set_ylabel('Cases')
ax1.grid(True, alpha=0.3)

# Hospitalizations
ax2.plot(covid_germany['date'], covid_germany['hospitalizations'], 
         marker='s', color='orange', linewidth=2)
ax2.set_title('COVID-19 Hospitalizations - Germany 2024', fontsize=14, fontweight='bold')
ax2.set_ylabel('Hospitalizations')
ax2.set_xlabel('Date')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 3.2 Measles Data Example

In [ ]:
# Fetch measles data for multiple countries
countries = ["DE", "FR", "IT", "ES"]
measles_data = []

for country in countries:
    try:
        data = epipulse.get_cases(
            disease="Measles",
            country=country,
            year=2024
        )
        measles_data.append(data)
    except Exception as e:
        print(f"Error fetching data for {country}: {e}")

# Combine all data
if measles_data:
    measles_combined = pd.concat(measles_data, ignore_index=True)
    print(f"\nTotal records: {len(measles_combined)}")
    print("\nData summary by country:")
    print(measles_combined.groupby('country')['cases'].sum())
else:
    print("No data retrieved")

In [ ]:
# Plot measles cases by country
if measles_data:
    plt.figure(figsize=(12, 6))
    
    for country in measles_combined['country'].unique():
        country_data = measles_combined[measles_combined['country'] == country]
        plt.plot(country_data['date'], country_data['cases'], 
                marker='o', label=country, linewidth=2)
    
    plt.title('Measles Cases by Country - 2024', fontsize=14, fontweight='bold')
    plt.xlabel('Date')
    plt.ylabel('Cases')
    plt.legend(title='Country')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

### 3.3 Influenza Surveillance

In [ ]:
# Fetch influenza data
influenza_eu = epipulse.get_cases(
    disease="Influenza",
    region="EU",
    year=2024
)

print(f"Retrieved {len(influenza_eu)} records")

# Show summary statistics
print("\nSummary statistics:")
print(influenza_eu[['cases', 'hospitalizations', 'deaths']].describe())

## 4. Comparative Analysis

In [ ]:
# Compare multiple diseases in one country
diseases_to_compare = ["COVID-19", "Influenza", "RSV"]
country = "DE"

comparison_data = {}
for disease in diseases_to_compare:
    try:
        data = epipulse.get_cases(
            disease=disease,
            country=country,
            year=2024
        )
        comparison_data[disease] = data['cases'].sum()
    except Exception as e:
        print(f"Error fetching {disease}: {e}")

# Create comparison chart
if comparison_data:
    plt.figure(figsize=(10, 6))
    diseases = list(comparison_data.keys())
    cases = list(comparison_data.values())
    
    bars = plt.bar(diseases, cases, color=['#e74c3c', '#3498db', '#2ecc71'])
    plt.title(f'Respiratory Disease Comparison - Germany 2024', 
              fontsize=14, fontweight='bold')
    plt.ylabel('Total Cases')
    plt.xlabel('Disease')
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height):,}',
                ha='center', va='bottom', fontsize=11)
    
    plt.tight_layout()
    plt.show()

## 5. Surveillance Summary

In [ ]:
# Get surveillance summary
summary = epipulse.get_surveillance_summary(country="Germany", year=2024)

print("EpiPulse Surveillance Summary")
print("="*60)
for key, value in summary.items():
    print(f"{key.replace('_', ' ').title()}: {value}")

## 6. Export Data

In [ ]:
# Export data to CSV
output_file = "epipulse_covid_germany_2024.csv"
covid_germany.to_csv(output_file, index=False)

print(f"✅ Data exported to: {output_file}")
print(f"File size: {Path(output_file).stat().st_size / 1024:.2f} KB")

## Summary

This notebook demonstrated:

1. ✅ Initializing the EpiPulse accessor
2. ✅ Listing available diseases (50+) and countries (30+)
3. ✅ Fetching surveillance data for COVID-19, Measles, and Influenza
4. ✅ Visualizing trends over time
5. ✅ Comparing data across countries and diseases
6. ✅ Exporting data for further analysis

### Next Steps

- Obtain an EpiPulse API key for full data access
- Explore event-based surveillance data
- Correlate with other data sources (climate, demographics)
- Build predictive models for disease trends

### Resources

- EpiPulse Portal: https://epipulse.ecdc.europa.eu/
- ECDC Website: https://www.ecdc.europa.eu/en/publications-data/epipulse
- API Documentation: Available through EpiPulse portal (requires registration)